# Supervised Fine-tuning Mistral 7B Instruct v1.0 for FinFact Dataset

In [10]:
import os
import random
import numpy as np
import pandas as pd
from datasets import Dataset

import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from sklearn.metrics import accuracy_score, classification_report

In [2]:
torch_version = torch.__version__
if torch_version == "2.0.1+cu118":
    print(f"Torch version is satisfied: {torch.__version__}")
else:
    print("Torch version should be 2.0.1+cu118. Please ensure that before going further")

Torch version is satisfied: 2.0.1+cu118


In [3]:
test = pd.read_csv("data/test_df.csv")

## vLLM Inference Server Engine for increased inference throughput and latency

In [4]:
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [5]:
finetuned_name = "skshreyas714/finfact-classifier-noise"

llm = LLM(model=finetuned_name, tensor_parallel_size=1)

INFO 10-19 05:35:47 llm_engine.py:72] Initializing an LLM engine with config: model='skshreyas714/finfact-classifier-noise', tokenizer='skshreyas714/finfact-classifier-noise', tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, quantization=None, seed=0)
INFO 10-19 05:36:03 llm_engine.py:207] # GPU blocks: 1330, # CPU blocks: 2048


In [ ]:
import random

ids = [i for i in range(len(test))]
idx = random.choice(ids)
print(idx)

ground_truth = test.iloc[idx]["label"]
prompt = test.iloc[idx]["text"]

print(f"Actual Output: {ground_truth}")

In [6]:
test_prompts = test["text"].tolist()
test_gts = test["label"].tolist()

In [7]:
sampling_params = SamplingParams(temperature=0, max_tokens=2, presence_penalty=1.5,
                                 skip_special_tokens=True, use_beam_search=False, early_stopping=False)

outputs = llm.generate(test_prompts, sampling_params)

predicted = []

# Print the outputs.
for output in outputs:
    generated_text = output.outputs[0].text.strip(" ")
    predicted.append(generated_text)
    # print(f"Generated text: {generated_text!r}")

Processed prompts: 100%|██████████| 357/357 [02:33<00:00,  2.32it/s]


### Evaluation

In [8]:
def compute_metrics(pred, labels):
    accuracy = accuracy_score(y_true=labels, y_pred=pred)
    return {"accuracy": accuracy}

In [12]:
# lower case the predictions for consistency
predicted = [i.lower() for i in predicted]

mistral_results = compute_metrics(predicted, test_gts)
mistral_accuracy = mistral_results["accuracy"]*100

print(f"Accuracy score from Mistral-7B --> {mistral_accuracy} %")

label_names = ['true', 'false', 'neutral']
print(f"Classification Report: \n {classification_report(test_gts, predicted, target_names=label_names)}")

Accuracy score from Mistral-7B --> 91.31652661064426 %
Classification Report: 
               precision    recall  f1-score   support

        true       0.91      1.00      0.95       126
       false       0.69      0.89      0.78        45
     neutral       0.99      0.86      0.92       186

    accuracy                           0.91       357
   macro avg       0.87      0.92      0.88       357
weighted avg       0.93      0.91      0.92       357

